Dong Woon Kim, M.D.  
BRUIN ID: 21399491  

Bellevue University  
Masters in Data Science  
DSC 550 Data Mining  

**Assignment Instructions**:

Download the labeled training dataset from this link: [Bag of Words Meets Bags of Popcorn](https://www.kaggle.com/c/word2vec-nlp-tutorial/data). 

**Part 1**: Using the TextBlob Sentiment Analyzer

1. Import the movie review data as a data frame and ensure that the data is loaded properly.
2. How many of each positive and negative reviews are there?
3. Use TextBlob to classify each movie review as positive or negative. Assume that a polarity score greater than or equal to zero is a positive sentiment and less than 0 is a negative sentiment.
4. Check the accuracy of this model. Is this model better than random guessing?
5. For up to five points extra credit, use another prebuilt text sentiment analyzer, e.g., VADER, and repeat steps (3) and (4).

Additional Comments

- The bag-of-words and tf-idf matrices are stored as sparse matrices because most entries are zero.  
- Each row in the bag-of-words/tf-idf matrices corresponds to a movie review.  
- The columns in the bag-of-words/tf-idf matrices correspond to unique words appearing in the movie reviews.  
- Entries in the bag-of-words matrix are the number of times a word appears in a review.  
- Entries in the tf-idf matrix are numbers representing the word importance in a review.  
- The bag-of-words/tf-idf matrices are possible feature (input) matrices for model building.  
- We will revisit this preprocessed text data to build a custom model in the future.  


Import the movie review data as a data frame and ensure that the data is loaded properly.    

In [29]:
import pandas as pd

df = pd.read_csv("./data/labeled_train_data.csv", encoding = "utf-8")
df = df.drop(columns = ["Unnamed: 0"], errors = "ignore").copy()
df.head()

,id,sentiment,review
0,5814_8,1,With all this stuff going down at the moment w...
1,2381_9,1,"\The Classic War of the Worlds\"" by Timothy Hi..."
2,7759_3,0,The film starts with a manager (Nicholas Bell)...
3,3630_4,0,It must be assumed that those who praised this...
4,9495_8,1,Superbly trashy and wondrously unpretentious 8...


How many of each positive and negative reviews are there?

In [30]:
positive = df[df["sentiment"] == 1]
negative = df[df["sentiment"] == 0]
print("Positive sentiment: ", len(positive))
print("Negative sentiment: ",len(negative))

# print(df["sentiment"].value_counts())

Positive sentiment:  12500
Negative sentiment:  12500


Use TextBlob to classify each movie review as positive or negative. Assume that a polarity score greater than or equal to zero is a positive sentiment and less than 0 is a negative sentiment.

In [31]:
from textblob import TextBlob

def predict_sentiment(review):
    polarity = TextBlob(review).sentiment.polarity
    return 1 if polarity >= 0 else 0

df["prediction"] = df["review"].apply(predict_sentiment)
df.head()

,id,sentiment,review,prediction
0,5814_8,1,With all this stuff going down at the moment w...,1
1,2381_9,1,"\The Classic War of the Worlds\"" by Timothy Hi...",1
2,7759_3,0,The film starts with a manager (Nicholas Bell)...,0
3,3630_4,0,It must be assumed that those who praised this...,1
4,9495_8,1,Superbly trashy and wondrously unpretentious 8...,0


Check the accuracy of this model. Is this model better than random guessing?

In [32]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

accuracy = accuracy_score(df["sentiment"], df["prediction"])
print("Accuracy:", accuracy)
print(confusion_matrix(df["sentiment"], df["prediction"]))
print(classification_report(df["sentiment"], df["prediction"]))
print("\nRandom guessing on this balanced dataset would be about 50% accuracy.")

Accuracy: 0.68524
[[ 5307  7193]
 [  676 11824]]
              precision    recall  f1-score   support

           0       0.89      0.42      0.57     12500
           1       0.62      0.95      0.75     12500

    accuracy                           0.69     25000
   macro avg       0.75      0.69      0.66     25000
weighted avg       0.75      0.69      0.66     25000


Random guessing on this balanced dataset would be about 50% accuracy.


For up to five points extra credit, use another prebuilt text sentiment analyzer, e.g., VADER, and repeat steps (3) and (4).

In [33]:
import nltk
#nltk.download("vader_lexicon")

from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def vader_predict(review):
    score = sia.polarity_scores(review)["compound"]
    return 1 if score >= 0 else 0

df["vader_prediction"] = df["review"].apply(vader_predict)

accuracy = accuracy_score(df["sentiment"], df["vader_prediction"])
print("VADER Accuracy:", accuracy)
print(confusion_matrix(df["sentiment"], df["vader_prediction"]))
print(classification_report(df["sentiment"], df["vader_prediction"]))

VADER Accuracy: 0.69356
[[ 6682  5818]
 [ 1843 10657]]
              precision    recall  f1-score   support

           0       0.78      0.53      0.64     12500
           1       0.65      0.85      0.74     12500

    accuracy                           0.69     25000
   macro avg       0.72      0.69      0.69     25000
weighted avg       0.72      0.69      0.69     25000



**Part 2**: Prepping Text for a Custom Model

If you want to run your own model to classify text, it needs to be in proper form to do so. The following steps will outline a procedure to do this on the movie reviews text.

1. Convert all text to lowercase letters.
2. Remove punctuation and special characters from the text.
3. Remove stop words.
4. Apply NLTK’s PorterStemmer.
5. Create a bag-of-words matrix from your stemmed text (output from (4)), where each row is a word-count vector for a single movie review (see sections 5.3 & 6.8 in the Machine Learning with Python Cookbook). Display the dimensions of your bag-of-words matrix. The number of rows in this matrix should be the same as the number of rows in your original data frame.
6. Create a term frequency-inverse document frequency (tf-idf) matrix from your stemmed text, for your movie reviews (see section 6.9 in the Machine Learning with Python Cookbook). Display the dimensions of your tf-idf matrix. These dimensions should be the same as your bag-of-words matrix.



In [34]:
import pandas as pd

df = pd.read_csv("./data/labeled_train_data.csv", encoding = "utf-8")
df = df.drop(columns = ["Unnamed: 0"], errors = "ignore").copy()
df.head()

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
stop_words = set(ENGLISH_STOP_WORDS)

def stem_review(review):
    words = re.sub(r"[^a-zA-Z]", " ", str(review).lower()).split()
    return " ".join(stemmer.stem(word) for word in words if word not in stop_words)

df["stemmed_review"] = df["review"].apply(stem_review)

In [35]:
df.head()

,id,sentiment,review,stemmed_review
0,5814_8,1,With all this stuff going down at the moment w...,stuff go moment mj ve start listen music watch...
1,2381_9,1,"\The Classic War of the Worlds\"" by Timothy Hi...",classic war world timothi hine entertain film ...
2,7759_3,0,The film starts with a manager (Nicholas Bell)...,film start manag nichola bell give welcom inve...
3,3630_4,0,It must be assumed that those who praised this...,assum prais film greatest film opera didn t re...
4,9495_8,1,Superbly trashy and wondrously unpretentious 8...,superbl trashi wondrous unpretenti s exploit h...


## Bag-of-Words

In [36]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer()
bow = count_vectorizer.fit_transform(df["stemmed_review"])

print("Bag-of-Words shape:", bow.shape)

Bag-of-Words shape: (25000, 49571)


## TF-IDF

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf = tfidf_vectorizer.fit_transform(df["stemmed_review"])

print("TF-IDF shape:", tfidf.shape)

TF-IDF shape: (25000, 49571)
